# 🛡️ Transaction Fraud Scorer Agentic AI (Google Colab Setup & Run)

Welcome to the **Transaction Fraud Scorer Agentic AI** interactive Colab notebook!

This notebook demonstrates how to:
1. Install and run a local **Ollama** server directly inside Google Colab (CPU or free T4 GPU tier).
2. Pull the open-source **`llama3.2:3b`** model (fast, lightweight, with native tool-calling).
3. Set up **Model Context Protocol (MCP)** fraud scoring tools (Velocity, Geolocation Anomaly, Merchant Risk, Account History).
4. Orchestrate an autonomous AI Agent using **LangChain (`ChatOllama`)** that dynamically investigates transactions, executes MCP tools, and produces explainable risk assessments (`APPROVE`, `REVIEW`, `DECLINE`).

## Step 1: Verify Hardware Environment
Check if a GPU is assigned. If running on CPU, `llama3.2:3b` (and `llama3.2:1b`) will still run smoothly!

In [1]:
!nvidia-smi || echo "Running in CPU mode. Lightweight models like llama3.2:3b run well on CPU too!"

Mon Oct 12 20:22:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Step 2: Install Ollama in Colab & Launch Background Server
We install Ollama using its official one-line script and start the background daemon process on `http://localhost:11434`.

In [2]:
import subprocess
import time

import requests

print("📥 Updating and installing zstd...")
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install zstd

print("📥 Downloading and installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

print("🚀 Starting Ollama daemon in background...")
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# Check health
for i in range(10):
    try:
        resp = requests.get("http://localhost:11434/api/version")
        if resp.status_code == 200:
            print(f"✅ Ollama server is UP and ready! Version: {resp.json().get('version')}")
            break
    except Exception:
        time.sleep(1)
else:
    print("⚠️ Ollama is still initializing. Please wait a moment.")

📥 Downloading and installing Ollama...
>>> Installing ollama to /usr/local
ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd
🚀 Starting Ollama daemon in background...


FileNotFoundError: [Errno 2] No such file or directory: 'ollama'

## Step 3: Pull Open-Source Model (`llama3.2:3b`)
Download `llama3.2:3b` (~2.0 GB). It provides high reasoning quality and native tool calling.

In [ ]:
print("📦 Pulling llama3.2:3b model into Ollama...")
!ollama pull llama3.2:3b

print("\n📋 Available local models:")
!ollama list

## Step 4: Clone Repository & Install Dependencies
Clone the project repository and install required packages (`langchain`, `langchain-ollama`, `mcp`, `pydantic`, `rich`).

In [ ]:
import os
import sys

# Clone repository if running in fresh Colab session
if not os.path.exists("TransactionFraudScorerAgenticAI"):
    !git clone https://github.com/daknamsubho/TransactionFraudScorerAgenticAI.git
    %cd TransactionFraudScorerAgenticAI
else:
    %cd TransactionFraudScorerAgenticAI

# Install project dependencies
!pip install -q "langchain>=1.4.0" "langchain-ollama>=1.1.0" "mcp>=1.3.0"
!pip install -q "pydantic>=2.7.0" "pydantic-settings>=2.2.0" "rich>=13.7.0" "httpx>=0.27.0"
!pip install -q -e .

if "src" not in sys.path:
    sys.path.append("src")

print("✅ Environment and dependencies configured successfully!")

## Step 5: Test MCP Risk Tools & LangChain Tool Adapters
Verify that the risk evaluation tools (Velocity, Geolocation, Merchant MCC, Account History) are properly loaded.

In [ ]:
from fraud_scorer.agent.tools import get_langchain_fraud_tools

tools = get_langchain_fraud_tools()
print(f"Loaded {len(tools)} LangChain MCP Tool Adapters:")
for t in tools:
    summary = t.description.strip().splitlines()[0]
    print(f"  • {t.name}: {summary}")

# Run a test invocation of velocity check
test_velocity = tools[0].invoke({"account_id": "acc_normal_01", "window_minutes": 60})
print("\nSample Tool Invocation Output:", test_velocity)

## Step 6: Initialize LangChain Agent Orchestrator with `ChatOllama`
Initialize `FraudScorerOrchestrator` pointing to the live Ollama daemon.

In [ ]:
from fraud_scorer.agent.orchestrator import FraudScorerOrchestrator, display_assessment, is_ollama_available
from fraud_scorer.models.transaction import GeoLocation, MerchantInfo, PaymentMethod, Transaction

print("Ollama server connected:", is_ollama_available("http://localhost:11434"))

# Initialize Orchestrator backed by ChatOllama
orchestrator = FraudScorerOrchestrator(
    ollama_base_url="http://localhost:11434", model_name="llama3.2:3b", provider="ollama"
)
print(f"✅ Fraud Scorer Orchestrator initialized using {orchestrator.model_name}!")

## Step 7: Scenario 1 - Normal Local Coffee Purchase
Evaluate a legitimate transaction ($42.50 in San Francisco at Blue Bottle Coffee by an established account).
Expected outcome: **`APPROVE`** with low composite risk score.

In [ ]:
tx_normal = Transaction(
    account_id="acc_normal_01",
    amount=42.50,
    currency="USD",
    location=GeoLocation(latitude=37.7749, longitude=-122.4194, city="San Francisco", country="US"),
    payment_method=PaymentMethod(payment_type="credit_card", card_last_four="4242", is_card_present=True),
    merchant=MerchantInfo(merchant_id="merch_cafe_88", name="Blue Bottle Coffee", mcc="5499"),
)

print("🔍 Evaluating Scenario 1 with LangChain & Ollama...")
assessment_1 = await orchestrator.evaluate_transaction(tx_normal)
display_assessment(tx_normal, assessment_1)

## Step 8: Scenario 2 - Multi-Vector Fraudulent Transaction
Evaluate a high-risk transaction ($3,500.00 in Tokyo at a casino shortly after London activity with high velocity and new untrusted account profile).
Expected outcome: **`DECLINE`** with triggered critical risk factors (impossible travel, gambling MCC, burst velocity).

In [ ]:
tx_suspicious = Transaction(
    account_id="acc_suspicious_01",
    amount=3500.00,
    currency="USD",
    location=GeoLocation(latitude=35.6762, longitude=139.6503, city="Tokyo", country="JP"),
    payment_method=PaymentMethod(payment_type="credit_card", card_last_four="9911", is_card_present=False),
    merchant=MerchantInfo(merchant_id="merch_casino_77", name="Tokyo High Roller Casino", mcc="7995"),
)

print("🔍 Evaluating Scenario 2 with LangChain & Ollama...")
assessment_2 = await orchestrator.evaluate_transaction(tx_suspicious)
display_assessment(tx_suspicious, assessment_2)

## Step 9: Interactive Custom Transaction Evaluator
You can test any custom scenario by modifying the transaction parameters below and re-running the cell!

In [ ]:
custom_tx = Transaction(
    account_id="acc_normal_01",
    amount=2400.00,
    currency="USD",
    location=GeoLocation(latitude=40.7128, longitude=-74.0060, city="New York", country="US"),
    payment_method=PaymentMethod(payment_type="credit_card", card_last_four="1234", is_card_present=False),
    merchant=MerchantInfo(merchant_id="merch_luxury_goods", name="Fifth Avenue Electronics", mcc="5732"),
)

print("🔍 Evaluating Custom Transaction...")
custom_assessment = await orchestrator.evaluate_transaction(custom_tx)
display_assessment(custom_tx, custom_assessment)